In [115]:
from langchain_core.tools import tool , InjectedToolArg
from langchain_openai import ChatOpenAI
import requests
from langchain_core.messages import HumanMessage
from typing import Annotated
import json

In [116]:
@tool
def get_conversion_factor(base_currency:str , target_currency:str)->float:
    """This function fetches the currency conversion factor between a base currency and a target currency"""
    url = f"https://v6.exchangerate-api.com/v6/81d4b5b441cc48c62ffb0b80/pair/{base_currency}/{target_currency}"
    response = requests.get(url)

    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [117]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [118]:
get_conversion_factor.invoke({"base_currency":"USD", "target_currency":"EUR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1775433602,
 'time_last_update_utc': 'Mon, 06 Apr 2026 00:00:02 +0000',
 'time_next_update_unix': 1775520002,
 'time_next_update_utc': 'Tue, 07 Apr 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'EUR',
 'conversion_rate': 0.8683}

In [119]:
convert.invoke({"base_currency_value": 100, "conversion_rate": 93.2})

9320.0

In [120]:
#Tool Binding
llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [121]:
llm_with_tools

RunnableBinding(bound=ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001C3718D3010>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001C3718B7F90>, root_client=<openai.OpenAI object at 0x000001C3718B7150>, root_async_client=<openai.AsyncOpenAI object at 0x000001C3718B6310>, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_conversion_factor', 'description': 'This function fetches the currency conversion fac

In [122]:
messages = [HumanMessage("What is the conversion factor between USD and INR , and based on that can you convert 575USD to INR?")]


In [123]:
ai_message = llm_with_tools.invoke(messages)

In [124]:
ai_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 122, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DRfnTfwbQBUteoLNFwmPAUXo7nKvx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6353-48af-7181-bdc5-d1efded49f4e-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_eFXRmph33K94BZD2JesXgshW', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 575}, 'id': 'call_CFNiq7IlPMpIQfnMUEIQhkA1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 122, 'output_tokens'

In [125]:
messages.append(ai_message)

In [126]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_eFXRmph33K94BZD2JesXgshW',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 575},
  'id': 'call_CFNiq7IlPMpIQfnMUEIQhkA1',
  'type': 'tool_call'}]

In [127]:
for tool_call in ai_message.tool_calls:
    #execute the first tool and get the first tool call and then the second tool
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        #fetch the conversion rate
        conversion_rates = json.loads(tool_message1.content)['conversion_rate']
        #aapend this tool message to messages
        messages.append(tool_message1)
    
    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rates
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [128]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR , and based on that can you convert 575USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 122, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DRfnTfwbQBUteoLNFwmPAUXo7nKvx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6353-48af-7181-bdc5-d1efded49f4e-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_eFXRmph33K94BZD2JesXgshW', 'type': 'tool_call'}, {'name': 'convert', 'ar

In [129]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 93.1258. \n\nBased on this conversion factor, 575 USD is equivalent to approximately 53,547.34 INR.'